<center><h1>Chen_Sian_HW3</h1></center>
<br>
<br>

Name: Sian Chen
<br>
Github Username: Ryan6546ea
<br>
USC ID: 3132413754

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

Get the AReM Data Set

In [2]:
files = sorted(Path("../data/AReM").glob("*/*.csv"))

### (b) Test and Train Data

In [3]:
test_files = [
    f for f in files
    if int(f.stem[7:]) <= (2 if f.parent.name.startswith("bending") else 3)
]

train_files = [f for f in files if f not in test_files]

### (c) Feature Extraction

#### i. Research

Common time-domain features include minimum, maximum, mean, median, variance, standard deviation, quartiles, skewness, kurtosis, root mean square, energy, and autocorrelation.

#### ii. Extraction

In [4]:
checks = []

for f in files:
    try:
        raw = pd.read_csv(f, comment="#", header=None)
        invalid = raw.apply(pd.to_numeric, errors="coerce").isna().sum().sum()
        checks.append([f"{f.parent.name}/{f.name}", *raw.shape, invalid, ""])
    except pd.errors.ParserError as error:
        checks.append([
            f"{f.parent.name}/{f.name}", None, None, None, str(error)
        ])

checks = pd.DataFrame(checks, columns=[
    "File", "Rows", "Columns", "Missing/non-numeric", "Read error"
])

checks[
    (checks["Rows"] != 480)
    | (checks["Columns"] != 7)
    | (checks["Missing/non-numeric"] > 0)
    | (checks["Read error"] != "")
]

,File,Rows,Columns,Missing/non-numeric,Read error
10,bending2/dataset4.csv,480.0,1.0,480.0,
18,cycling/dataset14.csv,NaN,NaN,NaN,Error tokenizing data. C error: Expected 7 fie...
27,cycling/dataset9.csv,NaN,NaN,NaN,Error tokenizing data. C error: Expected 7 fie...
56,sitting/dataset8.csv,479.0,7.0,0.0,


In [5]:
f = Path("../data/AReM/bending2/dataset4.csv")
lines = f.read_text().splitlines()

print(repr(next(
    line for line in lines
    if line.strip() and not line.startswith("#")
)))

'0 32.50 0.50 0.00 0.00 19.00 1.00 '


In [6]:
print(checks.loc[
    checks["Read error"] != "", ["File", "Read error"]
].to_string(index=False))

                 File                                                             Read error
cycling/dataset14.csv Error tokenizing data. C error: Expected 7 fields in line 485, saw 8\n
 cycling/dataset9.csv Error tokenizing data. C error: Expected 7 fields in line 485, saw 8\n


In [7]:
for name in ["cycling/dataset14.csv", "cycling/dataset9.csv"]:
    lines = (Path("../data/AReM") / name).read_text().splitlines()
    print(name, repr(lines[484]))

cycling/dataset14.csv '119750,34.00,2.12,24.50,1.80,22.25,2.59,'
cycling/dataset9.csv '119750,38.33,0.94,15.25,2.17,20.33,1.25,'


In [8]:
raw = pd.read_csv(
    "../data/AReM/sitting/dataset8.csv", comment="#", header=None
)

intervals = pd.DataFrame({
    "Previous time": raw[0].shift(),
    "Current time": raw[0],
    "Interval": raw[0].diff()
})

print(intervals["Interval"].value_counts())

intervals[
    intervals["Interval"].notna()
    & (intervals["Interval"] != intervals["Interval"].mode()[0])
]

Interval
250.0    477
500.0      1
Name: count, dtype: int64


,Previous time,Current time,Interval
54,13250.0,13750,500.0


The checks identified four files requiring attention. Inspection shows that
bending2/dataset4.csv uses whitespace delimiters, while cycling/dataset9.csv
and cycling/dataset14.csv contain an extra trailing comma on line 485.
These formatting issues are handled during reading.

sitting/dataset8.csv contains 479 observations, with the timestamp 13500
missing. I retain the available observations without imputation.

In [9]:
features = []

for f in files:
    data = pd.read_csv(
        f, comment="#", header=None, sep=r"[,\s]+",
        engine="python", usecols=range(1, 7)
    )
    features.append(data.describe().loc[
        ["min", "max", "mean", "50%", "std", "25%", "75%"]
    ].T.to_numpy().ravel())

features = pd.DataFrame(
    features,
    columns=[
        f"{s}_{i}"
        for i in range(1, 7)
        for s in ["min", "max", "mean", "median", "std", "q1", "q3"]
    ],
    index=range(1, len(files) + 1)
)
features.index.name = "Instance"

features

,min_1,max_1,mean_1,median_1,std_1,q1_1,q3_1,min_2,max_2,mean_2,...,std_5,q1_5,q3_5,min_6,max_6,mean_6,median_6,std_6,q1_6,q3_6
Instance,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.624792,40.500,1.476967,39.25,42.0000,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.430,0.582915,0.00,1.3000
2,38.00,45.67,42.812812,42.500,1.435550,42.00,43.6700,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.430,0.601010,0.00,1.3000
3,35.00,47.40,43.954500,44.330,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.430,0.513506,0.00,0.9400
4,33.00,47.75,42.179812,43.500,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.500,0.524317,0.00,1.0000
5,33.00,45.75,41.678063,41.750,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.430,0.389164,0.00,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,20.75,46.25,34.763333,35.290,4.742208,31.67,38.2500,0.0,12.68,4.223792,...,3.174681,14.2500,18.33,0.0,9.39,3.288271,3.270,1.647528,2.05,4.3050
85,21.50,51.00,34.935812,35.500,4.645944,32.00,38.0625,0.0,12.21,4.115750,...,3.192058,14.2375,18.25,0.0,10.21,3.280021,3.015,1.700918,2.12,4.5000
86,18.33,47.67,34.333042,34.750,4.948770,31.25,38.0000,0.0,12.48,4.396958,...,3.000493,13.7500,18.00,0.0,8.01,3.261583,2.980,1.617290,2.05,4.3200


#### iii. Standard Deviation

In [10]:
rng = np.random.default_rng(42)

bootstrap = pd.DataFrame([
    features.sample(frac=1, replace=True, random_state=rng).std()
    for _ in range(10000)
])

std_ci = pd.DataFrame({
    "Standard deviation": features.std(),
    "90% CI lower": bootstrap.quantile(0.05),
    "90% CI upper": bootstrap.quantile(0.95)
})

std_ci

,Standard deviation,90% CI lower,90% CI upper
min_1,9.569975,8.259076,10.776174
max_1,4.394362,3.324299,5.292799
mean_1,5.335718,4.695152,5.883563
median_1,5.440054,4.790869,6.007596
std_1,1.772153,1.570070,1.954889
q1_1,6.153590,5.564560,6.656036
q3_1,5.138925,4.311117,5.845088
min_2,0.000000,0.000000,0.000000
max_2,5.062729,4.625441,5.401985
mean_2,1.574164,1.397699,1.706804


I estimate each feature's sample standard deviation across the 88 instances.
I then bootstrap the instances 10,000 times, sampling 88 rows with replacement
each time. The 5th and 95th percentiles of the bootstrap standard deviations
form a 90% percentile confidence interval. A fixed random seed makes the
results reproducible.

#### iv. Select Features

I select mean, standard deviation, and maximum. Mean captures the overall
signal level, standard deviation measures variability, and maximum captures
peak values. Together, they summarize complementary aspects of the signal
that may help distinguish activities.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic model's training RSS is less than or equal to that of the linear
model. The linear model is a special case of the cubic model with
$\beta_2 = \beta_3 = 0$. Therefore, least squares cannot produce a larger
minimum training RSS when the additional terms are included.

### (b) Linear Test

We expect the linear model to have a lower test RSS. Since the true
relationship is linear, the additional quadratic and cubic terms are
unnecessary and can increase estimation variance without reducing
approximation bias. However, this is an expectation, not a guarantee for
every particular test set.

### (c) Not Linear Train

The cubic model's training RSS is still less than or equal to that of the
linear model. This follows from the fact that the cubic model contains
the linear model as a special case, regardless of the true relationship
between X and Y.

### (d) Not Linear Testing

There is not enough information to determine which model has a lower
test RSS. The cubic model may reduce bias by capturing nonlinearity, but
it also has higher estimation variance. If the bias reduction outweighs
the increase in variance, the cubic model may perform better; otherwise,
the linear model may perform better.

## 3. ISLR 3.7.3 - Extra Practice 

(a) Option iii is correct. Holding GPA and IQ fixed, the predicted salary
difference between college and high school graduates is

$$
\widehat{\text{Salary}}_{\text{college}}
-\widehat{\text{Salary}}_{\text{high school}}
=35-10\,\text{GPA}.
$$

Therefore, high school graduates have a higher predicted salary when
GPA > 3.5. The predicted salaries are equal when GPA = 3.5.

(b) For a college graduate, Level = 1. Substituting GPA = 4.0 and IQ = 110,

$$
\widehat{\text{Salary}}
=50+20(4)+0.07(110)+35(1)+0.01(4)(110)-10(4)(1)
=137.1.
$$

Since salary is measured in thousands of dollars, the predicted salary
is $137,100.

(c) False. A small estimated interaction coefficient does not necessarily
imply weak evidence of an interaction. Its magnitude depends on the
measurement scales of GPA and IQ. To assess statistical evidence, we
need its standard error and a corresponding t-test or confidence interval
for the null hypothesis that the interaction coefficient is zero.

## 3. ISLR 3.7.5 - Extra Practice 

For a fixed observation i, substituting the least squares estimate into
the fitted value gives

$$
\hat{y}_i
=x_i\hat{\beta}
=x_i\frac{\sum_{i'=1}^{n}x_{i'}y_{i'}}{\sum_{j=1}^{n}x_j^2}
=\sum_{i'=1}^{n}
\left(\frac{x_i x_{i'}}{\sum_{j=1}^{n}x_j^2}\right)y_{i'}.
$$

Therefore,

$$
a_{i'}=\frac{x_i x_{i'}}{\sum_{j=1}^{n}x_j^2},
$$

assuming $\sum_{j=1}^{n}x_j^2>0$. These coefficients depend only on the
predictor values, so each fitted value is a linear combination of the
observed response values.